In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#Load the dataset
df = pd.read_csv("../data/creditcard.csv")
print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
# Feature & Target Separation
X = df.drop('Class', axis=1)
y = df['Class']

In [6]:
# Train-Test Split(Stratified)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y)

In [7]:
# Create a 10% Stratified Sample for Tuning
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, 
    train_size=0.10, 
    random_state=42, 
    stratify=y_train
)

In [8]:
# Pipeline with Scaling, SMOTE and KNN

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()), # Scale features before SMOTE
    ('smote', SMOTE(random_state=42)), # Handle class imbalance
    ('knn', KNeighborsClassifier(algorithm='kd_tree')) # KNN model
])

In [ ]:
# Grid Search for Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
param_grid = {
    'knn__n_neighbors': [3, 5, 7],
    'knn__weights': ['uniform', 'distance']
}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_tune, y_tune)
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'knn__n_neighbors': 3, 'knn__weights': 'distance'}


In [19]:
# RE-FIT on the FULL Training Set

print("Re-fitting best model on 100% of the training data...")
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

Re-fitting best model on 100% of the training data...


Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                ('knn',
                 KNeighborsClassifier(algorithm='kd_tree', n_neighbors=3,
                                      weights='distance'))])

In [8]:
# 1. Create a tiny 5% sample of the TEST set
X_test_tiny, _, y_test_tiny, _ = train_test_split(
    X_test, y_test, 
    train_size=0.05, 
    random_state=42, 
    stratify=y_test
)

In [21]:
# 2. Predict on only the tiny sample and Run the report on tiny test sample
from sklearn.metrics import classification_report
print("Predicting on 5% of test data...")
y_pred_tiny = best_model.predict(X_test_tiny)

print(classification_report(y_test_tiny, y_pred_tiny))

Predicting on 5% of test data...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2843
           1       0.40      0.80      0.53         5

    accuracy                           1.00      2848
   macro avg       0.70      0.90      0.77      2848
weighted avg       1.00      1.00      1.00      2848



In [22]:
# Update the model to predict on the full test set
best_model.named_steps['knn'].n_jobs = -1
y_pred = best_model.predict(X_test)

In [20]:
# KNN Evaluation Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
recall = recall_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

Recall: 0.8469
Precision: 0.5461
F1-Score: 0.6640
ROC AUC: 0.9229


In [21]:
# Recall: 0.8469 - High: This means that the model correctly identifies 84.69% of the actual fraud cases.
# Precision: 0.5461 - Low: This means that only 54.61% of the predicted fraud cases are actually fraud. 
# This indicates a high false positive rate, which is common in imbalanced datasets.
# F1-Score: 0.6678 - This is the harmonic mean of precision and recall, indicating a balance between the two.
# ROC AUC: 0.9205 - This indicates that the model has a good ability to distinguish between the classes, 
# with a score of 0.9205 suggesting strong performance.

In [30]:
# Test Model on a Single Data Point (From Test Set)
single_data_point = X_test.iloc[[0]].values.reshape(1, -1)
single_prediction = best_model.predict(single_data_point)
print(f"Single Data Point Prediction: {single_prediction[0]}") 
actual_label = y_test.iloc[0]
print(f"Actual Label: {actual_label}")

Single Data Point Prediction: 0
Actual Label: 0


/Users/purushottamkumar/Documents/DSI/DSI_Project/DS-Finance1/venv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [35]:
# Analyze a True Positive Case
true_positive_indices = np.where((y_test == 1))[0]
print("Number of correctly detected frauds:", len(true_positive_indices))

index = true_positive_indices[0]
print(f"Testing on a single true positive data point at index: {index}")   

single_data_point = X_test.iloc[[index]]
actual_label = y_test.iloc[index]
prediction = best_model.predict(single_data_point)[0]

print("Actual Label:", actual_label)
print("Predicted Label:", prediction)
                                 

Number of correctly detected frauds: 98
Testing on a single true positive data point at index: 840
Actual Label: 1
Predicted Label: 1


In [36]:
# Analyze a True Positive Case at above index 840
single_data_point = X_test.iloc[[840]].values.reshape(1, -1)
single_prediction = best_model.predict(single_data_point)
print(f"Single Data Point Prediction: {single_prediction[0]}") 
actual_label = y_test.iloc[840]
print(f"Actual Label: {actual_label}")

Single Data Point Prediction: 1
Actual Label: 1


/Users/purushottamkumar/Documents/DSI/DSI_Project/DS-Finance1/venv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [37]:
# Analyze a case at different index 841
single_data_point = X_test.iloc[[841]].values.reshape(1, -1)
single_prediction = best_model.predict(single_data_point)
print(f"Single Data Point Prediction: {single_prediction[0]}") 
actual_label = y_test.iloc[841]
print(f"Actual Label: {actual_label}")

Single Data Point Prediction: 0
Actual Label: 0


/Users/purushottamkumar/Documents/DSI/DSI_Project/DS-Finance1/venv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [41]:
# Analyze False Positive Case - These are cases where the model predicted fraud (1) but the actual label is not fraud (0).
false_positive_indices = np.where((y_test == 0) & (y_pred == 1))[0]
print("Number of false positives:", len(false_positive_indices))

index = false_positive_indices[0]
print(f"Testing on a single false positive data point at index: {index}")

single_data_point = X_test.iloc[[index]].values.reshape(1, -1)
single_prediction = best_model.predict(single_data_point)
print(f"Single Data Point Prediction: {single_prediction[0]}") 
actual_label = y_test.iloc[index]
print(f"Actual Label: {actual_label}")


Number of false positives: 69
Testing on a single false positive data point at index: 165
Single Data Point Prediction: 1
Actual Label: 0


/Users/purushottamkumar/Documents/DSI/DSI_Project/DS-Finance1/venv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [9]:
# Precision: 0.5461 - Low: This means that only 54.61% of the predicted fraud cases are actually fraud. 
# This indicates a high false positive rate, which is common in imbalanced datasets.
# Lets Try with another KNN model with different hyperparameters to see if we can improve precision.

from sklearn.model_selection import GridSearchCV
param_grid = {
    'knn__n_neighbors': [9, 11, 13, 15],
    'knn__weights': ['uniform', 'distance']
}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='precision', n_jobs=-1)
grid_search.fit(X_tune, y_tune)
print("Best Parameters:", grid_search.best_params_)





Best Parameters: {'knn__n_neighbors': 9, 'knn__weights': 'distance'}


In [10]:
# RE-FIT on the FULL Training Set

print("Re-fitting best model on 100% of the training data...")
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

Re-fitting best model on 100% of the training data...


Pipeline(steps=[('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)),
                ('knn',
                 KNeighborsClassifier(algorithm='kd_tree', n_neighbors=9,
                                      weights='distance'))])

In [11]:
# 1. Create a tiny 5% sample of the TEST set
X_test_tiny, _, y_test_tiny, _ = train_test_split(
    X_test, y_test, 
    train_size=0.05, 
    random_state=42, 
    stratify=y_test
)

In [12]:
# Get the best parameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print("Best Parameters:", best_params)

# Predict on the test set using the best model
y_pred_best = best_model.predict(X_test_tiny)



Best Parameters: {'knn__n_neighbors': 9, 'knn__weights': 'distance'}


In [13]:
# Update the model to predict on the full test set
best_model.named_steps['knn'].n_jobs = -1
y_pred_best = best_model.predict(X_test)

In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

recall = recall_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
roc_auc = roc_auc_score(y_test, y_pred_best)

print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

Recall: 0.8878
Precision: 0.3537
F1-Score: 0.5058
ROC AUC: 0.9425


In [16]:
# Grid Search with precision and different knn_n_neighours value has not improved the precision score
# Hence our best model is the one with n_neighbors=3 and weights='uniform' which gives us a precision score of 0.5461.